In [1]:
import sys
!{sys.executable} -m pip install hdfs

In [6]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, concat, lit
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable

builder = SparkSession.builder \
    .appName("Gold_Dim_Calendario") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .enableHiveSupport()
spark = configure_spark_with_delta_pip(builder).getOrCreate()

gold_calendario_path = "hdfs://hdfs-nn:9000/demo/gold/dim_calendario"

ano_inicio = 1900
ano_fim = 2025

print(f"A criar a Tabela Dimensão Calendário do ano {ano_inicio} até ao ano {ano_fim}\n")

dim_calendario = spark.range(ano_inicio, ano_fim + 1) \
    .withColumn("ano", col("id").cast("int")) \
    .withColumn("id_calendario", concat(lit("C"), col("id").cast("string"))) \
    .select("id_calendario", "ano")

print(f"-> A Gravar Dimensão Calendário em {gold_calendario_path}")
dim_calendario.repartition(1).write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(gold_calendario_path)

print(f"\n-> Tabela Dimensão Calendário")
dim_calendario.orderBy(col("ano").desc()).show(10)

spark.stop()

A criar a Tabela Dimensão Calendário do ano 1900 até ao ano 2025

-> A Gravar Dimensão Calendário em hdfs://hdfs-nn:9000/demo/gold/dim_calendario

-> Tabela Dimensão Calendário
+-------------+----+
|id_calendario| ano|
+-------------+----+
|        C2025|2025|
|        C2024|2024|
|        C2023|2023|
|        C2022|2022|
|        C2021|2021|
|        C2020|2020|
|        C2019|2019|
|        C2018|2018|
|        C2017|2017|
|        C2016|2016|
+-------------+----+
only showing top 10 rows

